In [1]:
!pip install -q timm kagglehub albumentations
import os
import kagglehub
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import pandas as pd
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Colab_Models/Grocery_Pretrain/'
os.makedirs(SAVE_DIR, exist_ok=True)

os.environ['KAGGLE_USERNAME'] = "aromannn"
os.environ['KAGGLE_KEY'] = "83ce27625be5b3ce50012271ac850e60"

BATCH_SIZE = 32
EPOCHS = 3
LR = 5e-4
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Устройство: {DEVICE}")
print(f"Сохраняем модель в: {SAVE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Устройство: cuda
Сохраняем модель в: /content/drive/MyDrive/Colab_Models/Grocery_Pretrain/


In [2]:
print("Скачиваем датасет (validmodel/grocery-store-dataset)...")
try:
    dataset_path = kagglehub.dataset_download('validmodel/grocery-store-dataset')
    print(f"УСПЕХ! Путь: {dataset_path}")
except Exception as e:
    print(f" Ошибка скачивания: {e}")

# Ищем папку 'dataset', где лежат train.txt и classes.csv
meta_dir = None
for root, dirs, files in os.walk(dataset_path):
    if 'train.txt' in files and 'classes.csv' in files:
        meta_dir = root
        break

if not meta_dir:
    raise ValueError("Не найдены служебные файлы датасета!")

# Корень с изображениями находится на уровень выше
img_root = os.path.dirname(meta_dir)
print(f" Корневая папка с картинками: {img_root}")
print(f" Папка с разметкой: {meta_dir}")


Скачиваем датасет (validmodel/grocery-store-dataset)...


100%|██████████| 118M/118M [00:01<00:00, 69.2MB/s]

Extracting files...


УСПЕХ! Путь: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19
 Корневая папка с картинками: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset
 Папка с разметкой: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset


In [3]:
# Посмотрим первые строки и их "сырой" вид
with open(os.path.join(meta_dir, 'train.txt'), 'r') as f:
    for i in range(5):
        line = f.readline()
        print(repr(line))


'train/Fruit/Apple/Golden-Delicious/Golden-Delicious_001.jpg, 0, 0\n'
'train/Fruit/Apple/Golden-Delicious/Golden-Delicious_002.jpg, 0, 0\n'
'train/Fruit/Apple/Golden-Delicious/Golden-Delicious_003.jpg, 0, 0\n'
'train/Fruit/Apple/Golden-Delicious/Golden-Delicious_004.jpg, 0, 0\n'
'train/Fruit/Apple/Golden-Delicious/Golden-Delicious_005.jpg, 0, 0\n'


In [4]:
print(f"img_root = {img_root}")
print(f"\n Что находится в img_root:")
for item in sorted(os.listdir(img_root)):
    full = os.path.join(img_root, item)
    kind = "📁" if os.path.isdir(full) else "📄"
    print(f"  {kind} {item}")

train_folder = os.path.join(img_root, 'train')
if os.path.exists(train_folder):
    print(f"\n Папка 'train' ЕСТЬ: {train_folder}")
    print("Содержимое train:")
    for item in sorted(os.listdir(train_folder))[:5]:
        print(f"  {item}")
else:
    print(f"\nПапки 'train' НЕТ в img_root")

    print("\nИщем первые jpg файлы по всему датасету...")
    count = 0
    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if f.endswith('.jpg'):
                print(f"  Найден: {os.path.join(root, f)}")
                count += 1
                if count >= 5:
                    break
        if count >= 5:
            break


img_root = /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset

 Что находится в img_root:
  📁 dataset
  📁 sample_images

Папки 'train' НЕТ в img_root

Ищем первые jpg файлы по всему датасету...
  Найден: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset/val/Vegetables/Ginger/Ginger_001.jpg
  Найден: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset/val/Vegetables/Ginger/Ginger_003.jpg
  Найден: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset/val/Vegetables/Ginger/Ginger_002.jpg
  Найден: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset/val/Vegetables/Ginger/Ginger_004.jpg
  Найден: /root/.cache/kagglehub/datasets/validmodel/grocery-store-dataset/versions/19/GroceryStoreDataset/dataset/val/Vegetables/Pepper/Orange-Bell-Pep

In [5]:
classes_csv_path = os.path.join(meta_dir, 'classes.csv')
classes_df = pd.read_csv(classes_csv_path)

col_class_name = classes_df.columns[0]
col_fine_label = classes_df.columns[1]

label_to_name = dict(zip(classes_df[col_fine_label].astype(int), classes_df[col_class_name]))
NUM_CLASSES = len(label_to_name)

print(f"Найдено классов: {NUM_CLASSES}")

def load_split(txt_file):
    items = []
    filepath = os.path.join(meta_dir, txt_file)

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(',')
            if len(parts) >= 2:
                rel_path = parts[0].strip()
                fine_label = int(parts[1].strip())

                abs_path = os.path.join(meta_dir, rel_path)

                if os.path.exists(abs_path):
                    items.append((abs_path, fine_label))

    return items

train_items = load_split('train.txt')
val_items = load_split('val.txt')

print(f"Тренировочная выборка: {len(train_items)} фото")
print(f"Валидационная выборка: {len(val_items)} фото")

if len(train_items) == 0:
    print("ОШИБКА")


Найдено классов: 81
Тренировочная выборка: 2640 фото
Валидационная выборка: 296 фото


In [6]:

train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),  # Смена освещения
    A.GaussNoise(p=0.2),                # Шум камеры
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class GroceryDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        return img, label

train_dl = DataLoader(GroceryDataset(train_items, train_transforms), batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(GroceryDataset(val_items, val_transforms), batch_size=BATCH_SIZE, shuffle=False)

print("Данные готовы к загрузке.")


Данные готовы к загрузке.


In [ ]:
# Создаем модель на 81 класс
print(f"Инициализация tf_efficientnet_b4_ns для {NUM_CLASSES} классов...")
model = timm.create_model('tf_efficientnet_b4_ns', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_loss = float('inf')
save_path = os.path.join(SAVE_DIR, 'grocery_b4_pretrain.pth')

print(" Начинаем обучение...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for X, y in pbar:
        X, y = X.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss / len(train_dl)

    # Валидация
    model.eval()
    val_loss = 0.0
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in val_dl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss = criterion(logits, y)
            val_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_val_loss = val_loss / len(val_dl)
    val_acc = correct / total * 100

    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.1f}%")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Улучшение! Модель сохранена: {save_path}")

print("\nВспомогательное обучение завершено! Веса сохранены на Диск.")
print("Теперь можешь загрузить этот .pth файл в основной скрипт через strict=False.")


🚀 Инициализация tf_efficientnet_b4_ns для 81 классов...


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

⏳ Начинаем обучение (вспомогательный этап)...


Epoch 1/3 [Train]: 100%|██████████| 83/83 [00:45<00:00,  1.81it/s, loss=0.8895]


📈 Epoch 1 | Train Loss: 1.8854 | Val Loss: 1.3760 | Val Acc: 62.5%
💾 Улучшение! Модель сохранена: /content/drive/MyDrive/Colab_Models/Grocery_Pretrain/grocery_b4_pretrain.pth


Epoch 2/3 [Train]: 100%|██████████| 83/83 [00:48<00:00,  1.71it/s, loss=0.1776]


📈 Epoch 2 | Train Loss: 0.3207 | Val Loss: 1.0078 | Val Acc: 72.6%
💾 Улучшение! Модель сохранена: /content/drive/MyDrive/Colab_Models/Grocery_Pretrain/grocery_b4_pretrain.pth


Epoch 3/3 [Train]:  99%|█████████▉| 82/83 [00:46<00:00,  1.77it/s, loss=0.1592]